# CS415 Lab 4: Fine Tuning LLMs

This notebook demonstrates four parameter-efficient fine-tuning techniques:

1. Adapters (IA3)
2. BitFit  
3. LoRA  
4. Prefix-Tuning  

We will use the t5-small model for sentiment classification with the SST-2 dataset.

Example input: "sst2 sentence: This movie was great."  
Example output: "positive"

Because T5 is a text-to-text model, it can produce meaningful zero-shot predictions before training.  
This allows us to observe performance before and after fine-tuning.

We start by installing the necessary libraries:

In [ ]:
!pip install -q transformers datasets evaluate peft accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [ ]:
# Telling huggingface not to ask for API keys and turning off warnings
import os
import warnings

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"

warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import time
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer,
    DataCollatorForSeq2Seq, Trainer, TrainingArguments
)
from peft import (
    LoraConfig, PrefixTuningConfig, PeftModel,
    TaskType, get_peft_model, PeftConfig, IA3Config
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "t5-small"

# Loading the tokenizer associated with t5-small
tokenizer = AutoTokenizer.from_pretrained(model_name)


⚙️  Running in WANDB offline mode


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Next, we load our SST-2 dataset, tokenize and split it into train and validation sets and initialize the t5-small model.

In [ ]:
dataset = load_dataset("glue", "sst2")

def preprocess(ex):
    texts = ["sst2 sentence: " + s for s in ex["sentence"]]
    targets = ["positive" if l == 1 else "negative" for l in ex["label"]]

    model_inputs = tokenizer(texts, truncation=True)

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

ds = dataset.map(
    preprocess,
    batched=True,
    remove_columns=dataset["train"].column_names
)

train_ds = ds["train"]
val_ds   = ds["validation"]

baseline = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=baseline) # Required parameters are tokenizer and model

README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

We can get a summary of our dataset and look at the first example in the training set by running the code below:

In [ ]:
print(ds)
print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 872
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1821
    })
})
{'input_ids': [3, 7, 7, 17, 357, 7142, 10, 7387, 126, 2829, 2865, 45, 8, 21555, 3173, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [2841, 1]}


We define some helper functions

In [ ]:
# Function for calculating the accuracy
def manual_accuracy(predictions, labels, tokenizer):

    # Some HF versions return predictions as a tuple: (predictions,)
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.array(predictions)
    labels = np.array(labels)

    # If predictions are logits: (batch, seq_len, vocab)
    if predictions.ndim == 3:
        predictions = predictions.argmax(-1)

    # Convert numpy arrays to python lists
    predictions = predictions.tolist()
    labels = labels.tolist()

    # Replace -100 with pad_token_id to enable decoding
    pad = tokenizer.pad_token_id
    predictions = [[p if p != -100 else pad for p in pred] for pred in predictions]
    labels = [[l if l != -100 else pad for l in lab] for lab in labels]

    # Decode
    pred_texts = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    label_texts = tokenizer.batch_decode(labels, skip_special_tokens=True)

    pred_texts = [p.strip().lower() for p in pred_texts]
    label_texts = [l.strip().lower() for l in label_texts]

    # Compute exact match accuracy
    correct = sum(p == l for p, l in zip(pred_texts, label_texts)) # check the number of matches between pred_texts and label_texts

    return correct / len(pred_texts) # return accuracy


In [ ]:
# Function for counting the number of trainable parameters in the model
def count_trainable(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total, 100 * trainable / total


Unzipping the files containing the parameters of the fine-tuned models.

In [ ]:
!unzip adapter_model.zip -d adapter_model
!unzip bitfit_model.zip -d bitfit_model
!unzip lora_model.zip -d lora_model
!unzip prefix_model.zip -d prefix_model


Archive:  adapter_model.zip
  inflating: adapter_model/adapter_config.json  
  inflating: adapter_model/adapter_model.safetensors  
Archive:  bitfit_model.zip
  inflating: bitfit_model/config.json  
  inflating: bitfit_model/generation_config.json  
  inflating: bitfit_model/model.safetensors  
  inflating: bitfit_model/special_tokens_map.json  
  inflating: bitfit_model/spiece.model  
  inflating: bitfit_model/tokenizer.json  
  inflating: bitfit_model/tokenizer_config.json  
  inflating: bitfit_model/training_args.bin  
Archive:  lora_model.zip
  inflating: lora_model/adapter_config.json  
  inflating: lora_model/adapter_model.safetensors  
Archive:  prefix_model.zip
  inflating: prefix_model/adapter_config.json  
  inflating: prefix_model/adapter_model.safetensors  


## Baseline Zero-Shot Evaluation

We evaluate the zero-shot performance of the baseline T5 model.


In [ ]:
args = TrainingArguments(output_dir="out_baseline", per_device_eval_batch_size=32)

trainer = Trainer(
    model=baseline,
    args=args,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator
)

raw_pred = trainer.predict(val_ds)

preds = raw_pred.predictions
labels = raw_pred.label_ids

baseline_accuracy = manual_accuracy(preds, labels, tokenizer)
print(f"Baseline accuracy: {baseline_accuracy}")


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Baseline accuracy: 0.8967889908256881


As we can see, the baseline performance of the model is around 89.7% which is already high.

## 1. Adapters

Adapters insert small trainable modules into the model while freezing the original weights.

The T5 model natively supports adapters via the PEFT library.


In [ ]:
# Adapter configuration
adapter_cfg = IA3Config(
    task_type="SEQ_2_SEQ_LM"
)

# Building Adapter model
adapter_model = get_peft_model(
    AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device),
    adapter_cfg
)

a_trainable, a_total, a_pct = count_trainable(adapter_model) # count the number of trainable parameters using the helper function

# Training args
adapter_args = TrainingArguments(
    output_dir="out_adapter",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    learning_rate=1e-3,
    warmup_ratio=0.1,
    weight_decay=0.0,
    logging_steps=1000,
    save_strategy="no",          # no if using pretrained weights
)

# Creating collator
adapter_collator = DataCollatorForSeq2Seq(tokenizer, model=adapter_model)

adapter_trainer = Trainer(
    model=adapter_model,
    args=adapter_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=adapter_collator
)

# Training / loading
train_adapters = False # False if using pretrained weights
start = time.time()
if train_adapters:
    adapter_trainer.train()
    a_time = time.time() - start
    adapter_trainer.save_model("adapter_model")
else:
    a_time = None
    base = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
    adapter_model = PeftModel.from_pretrained(base, "adapter_model").to(device)

    # Rebuilding trainer with the newly loaded model
    adapter_trainer = Trainer(
        model=adapter_model,
        args=adapter_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        data_collator=adapter_collator
    )


# Evaluation
preds, labels = adapter_trainer.predict(val_ds)[:2]
adapter_accuracy = manual_accuracy(preds, labels, tokenizer)
print(f"Adapter accuracy: {adapter_accuracy}")


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Adapter accuracy: 0.9036697247706422


## 2. BitFit

BitFit trains only bias terms and keeps all other weights frozen.


In [ ]:
# Building BitFit model
bitfit_model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

# Freezing everything
for p in bitfit_model.parameters():
    p.requires_grad = False

# Unfreezing all biases + lm_head
for name, p in bitfit_model.named_parameters():
    if "bias" in name or "lm_head" in name:
        p.requires_grad = True

b_trainable, b_total, b_pct = count_trainable(bitfit_model)
print("Trainable params:", b_trainable, "/", b_total, f"({b_pct:.4f}%)")

# Training arguments
bitfit_args = TrainingArguments(
    output_dir="out_bitfit",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    learning_rate=5e-4,
    warmup_ratio=0.1,
    logging_steps=1000,
    save_strategy="no",
)

# Creating collator
bitfit_collator = DataCollatorForSeq2Seq(tokenizer, model=bitfit_model)

# Trainer
bitfit_trainer = Trainer(
    model=bitfit_model,
    args=bitfit_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=bitfit_collator,
)

# Training / loading
train_bitfit = False
start = time.time()
if train_bitfit:
    bitfit_trainer.train()
    b_time = time.time() - start
    bitfit_trainer.save_model("bitfit_model")

else:
    b_time = None
    # Loading the fine-tuned model
    bitfit_model = AutoModelForSeq2SeqLM.from_pretrained("bitfit_model").to(device)

    bitfit_trainer = Trainer(
        model=bitfit_model,
        args=bitfit_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        data_collator=adapter_collator
    )

# Evaluation
preds, labels = bitfit_trainer.predict(val_ds)[:2]
bitfit_accuracy = manual_accuracy(preds, labels, tokenizer)

print(f"BitFit accuracy: {bitfit_accuracy}")


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Trainable params: 512 / 60506624 (0.0008%)


BitFit accuracy: 0.8922018348623854


## 3. LoRA

LoRA injects low-rank matrices into attention layers while freezing the original weights.


In [ ]:
# LoRA Configuration
lora_cfg = LoraConfig(
    task_type="SEQ_2_SEQ_LM",
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q", "k", "v", "o", "wi", "wo"]
)

# Building LoRA model
lora_model = get_peft_model(
    AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device),
    lora_cfg
)

l_trainable, l_total, l_pct = count_trainable(lora_model)
print("Trainable:", l_trainable, "/", l_total, f"({l_pct:.4f}%)")

# Training arguments
lora_args = TrainingArguments(
    output_dir="out_lora",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    learning_rate=1e-3,
    warmup_ratio=0.1,
    weight_decay=0.0,
    logging_steps=1000,
    save_strategy="no",
)

# Creating collator
lora_collator = DataCollatorForSeq2Seq(tokenizer, model=lora_model)

# Trainer
lora_trainer = Trainer(
    model=lora_model,
    args=lora_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=lora_collator,
)

# Training / loading
train_lora = False
start = time.time()
if train_lora:
    lora_trainer.train()
    l_time = time.time() - start
    lora_trainer.save_model("lora_model")
else:
    l_time = None
    base = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
    lora_model = PeftModel.from_pretrained(base, "lora_model").to(device)

    lora_trainer = Trainer(
        model=lora_model,
        args=lora_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        data_collator=lora_collator,
        )

# Evaluation
preds, labels = lora_trainer.predict(val_ds)[:2]
lora_accuracy = manual_accuracy(preds, labels, tokenizer) # calculate accuracy using the helper function we defined

print(f"LoRA accuracy: {lora_accuracy}")


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Trainable: 1081344 / 61587968 (1.7558%)


LoRA accuracy: 0.9185779816513762


## 4. Prefix-Tuning

 Prefix-tuning learns special trainable embeddings prepended to the input



In [ ]:
# Prefix Tuning Configuration
prefix_cfg = PrefixTuningConfig(
    task_type="SEQ_2_SEQ_LM",
    num_virtual_tokens=20,
    encoder_hidden_size=512,     # hidden size for T5-small
    token_dim=512,               # embedding dimension
    num_attention_heads=8,       # T5-small has 8 heads
    num_layers=6                 # number of encoder/decoder layers
)

# Building Prefix Model
prefix_model = get_peft_model(
    AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device),
    prefix_cfg
)

p_trainable, p_total, p_pct = count_trainable(prefix_model)
print("Trainable:", p_trainable, "/", p_total, f"({p_pct:.4f}%)")

# Training arguments
prefix_args = TrainingArguments(
    output_dir="out_prefix",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    learning_rate=1e-3,          # prefix tuning benefits from higher LR
    warmup_ratio=0.1,
    logging_steps=1000,
    save_strategy="no",
)

# Creating collator
prefix_collator = DataCollatorForSeq2Seq(tokenizer, model=prefix_model)

# Trainer
prefix_trainer = Trainer(
    model=prefix_model,
    args=prefix_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=prefix_collator,
)

# Training / loading
train_prefix = False
start = time.time()
if train_prefix:
    prefix_trainer.train()
    p_time = time.time() - start
    prefix_trainer.save_model("prefix_model")

else:
    p_time = None
    base = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
    prefix_model = PeftModel.from_pretrained(base, "prefix_model").to(device)

    prefix_trainer = Trainer(
    model=prefix_model,
    args=prefix_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=prefix_collator,
)

# Evaluation
preds, labels = prefix_trainer.predict(val_ds)[:2]
prefix_accuracy = manual_accuracy(preds, labels, tokenizer)

print(f"Prefix accuracy: {prefix_accuracy}")


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Trainable: 122880 / 60629504 (0.2027%)


Prefix accuracy: 0.8899082568807339


## Summary of Results

Below we compare:
- accuracy
- percent of parameters updated
- training time
for all four PEFT methods.


In [ ]:
# These are the times that were measured during fine-tuning. If you ran the fine tuning code yourself
# instead of loading the weights directly, you can delete the line below and see your own measurements.
a_time, b_time, l_time, p_time = [364.08, 458.90, 938.32, 350.02]

summary = {
    "baseline": {"acc": baseline_accuracy, "pct": 0, "time": 0},
    "adapters": {"acc": adapter_accuracy, "pct": a_pct, "time": a_time},
    "bitfit": {"acc": bitfit_accuracy, "pct": b_pct, "time": b_time},
    "lora": {"acc": lora_accuracy, "pct": l_pct, "time": l_time},
    "prefix": {"acc": prefix_accuracy, "pct": p_pct, "time": p_time},
}

print("method     acc        params(%)     time(s)")
print("---------------------------------------------")

for method, stats in summary.items():
    print(f"{method:10} {stats['acc']:.4f}    {stats['pct']:8.4f}      {stats['time']:.2f}")



method     acc        params(%)     time(s)
---------------------------------------------
baseline   0.8968      0.0000      0.00
adapters   0.9037      0.0710      364.08
bitfit     0.8922      0.0008      458.90
lora       0.9186      1.7558      938.32
prefix     0.8899      0.2027      350.02


## Notes

As we can see, the baseline performance of the model was already around 89.7%, which is quite high for the SST-2 dataset. This is because the T5 model had already learned general sentiment patterns during pretraining. When a small model already performs near the task ceiling, PEFT methods can only provide small gains—and in some cases may even slightly reduce accuracy.

This behavior is expected. PEFT methods were designed for efficiently fine-tuning very large models (with billions of parameters), where full fine-tuning is too expensive. Our model, by contrast, has about 60.5 million parameters, so the benefits of PEFT are less dramatic. Still, methods like LoRA can provide a noticeable improvement, while others may show minimal or negative change because the task is already mostly solved by the baseline model.

In this lab we have:
1. Measured the baseline performance of the model  
2. Fine-tuned the model with each PEFT method  
3. Compared accuracy, parameter efficiency, and training time
